In [2]:
from stockpyl.supply_chain_network import serial_system
from stockpyl.ssm_serial import optimize_base_stock_levels


In [7]:
from stockpyl.supply_chain_node import SupplyChainNode
from stockpyl.supply_chain_network import SupplyChainNetwork
from stockpyl.rq import r_q_poisson_exact   # single‑node (r,Q)
from math import inf

# --------------------------
# 1. Build 3‑warehouse / 10‑store network
# --------------------------
net = SupplyChainNetwork()

# Warehouses (W1 is upstream, W2/W3 regional)
W1 = SupplyChainNode(index=1, name="W1",
                     local_holding_cost=0.5,    # $/unit/week
                     stockout_cost=0,           # assume no external penalty at W1
                     shipment_lead_time=2)      # weeks to W2/W3

W2 = SupplyChainNode(index=2, name="W2",
                     local_holding_cost=0.7,
                     stockout_cost=0,
                     shipment_lead_time=1)      # to its stores

W3 = SupplyChainNode(index=3, name="W3",
                     local_holding_cost=0.7,
                     stockout_cost=0,
                     shipment_lead_time=1)

net.add_node(W1); net.add_node(W2); net.add_node(W3)

# Stores S1–S10 (penalize stockouts here)
stores = {}
for i in range(4, 14):  # indices 4..13 = 10 stores
    stores[i] = SupplyChainNode(index=i,
                                name=f"S{i-3}",
                                local_holding_cost=1.0,  # more expensive to hold in store
                                stockout_cost=50.0,      # lost sales / rush cost per unit
                                shipment_lead_time=0)    # assume same‑day from WH
    net.add_node(stores[i])

# Connect network (tree topology)
# W1 → W2, W3
net.add_successor(W1, W2)
net.add_successor(W1, W3)

# W2 → S1–S5  (indices 4..8)
for idx in range(4, 9):
    net.add_successor(W2, stores[idx])

# W3 → S6–S10 (indices 9..13)
for idx in range(9, 14):
    net.add_successor(W3, stores[idx])

# --------------------------
# 2. Simple (r,Q) optimization for ONE store
#    (repeat per store in real project)
# --------------------------
# Suppose we have weekly Poisson demand estimate for store S1:
demand_mean_S1 = 40.0            # units/week
holding_cost_S1 = stores[4].local_holding_cost   # 1.0
stockout_penalty_S1 = stores[4].stockout_cost    # 50.0
order_cost_S1 = 100.0            # fixed cost per order from W2 to S1
purchase_cost_S1 = 10.0          # unit purchase cost (optional for rq function)
lead_time_S1 = 1.0
# Tune "service level" via penalty ratio; rq_poisson_exact API:
# r_q_poisson_exact(lmbda, fixed_cost, holding_cost, penalty_cost, purchase_cost=0)
r_S1, Q_S1, cost_S1 = r_q_poisson_exact(
    holding_cost=holding_cost_S1,
    stockout_cost=stockout_penalty_S1,
    fixed_cost=order_cost_S1,
    demand_mean=demand_mean_S1,
    lead_time=lead_time_S1
)

print("Store S1 (r,Q) policy:")
print("  r =", r_S1)
print("  Q =", Q_S1)
print("  expected cost per week ≈", cost_S1)

# --------------------------
# 3. Placeholder: multi‑echelon optimization for the whole network
# --------------------------
# Here you would set per‑node demand (at stores), lead times, and holding/stockout
# costs, then call a multi‑echelon optimizer such as meio_by_enumeration
# or coordinate descent on an InventoryNetwork instance. For now we just show
# where that step plugs in:

# from stockpyl.meio_general import meio_by_enumeration
# best_S, best_cost = meio_by_enumeration(network=net, truncation_lo=0, truncation_hi=100)
# print("Optimal echelon base stocks:", best_S)
# print("Optimal total cost:", best_cost)


Store S1 (r,Q) policy:
  r = 41
  Q = 93
  expected cost per week ≈ 94.9053082854795


In [8]:
# -------------------------- 
# 4. (r,Q) policies for ALL 10 stores
# --------------------------
store_policies = {}
for store_idx, store in stores.items():
    # Realistic variation across stores
    demand_mean = 30 + store_idx * 2  # S1:32, S2:34, ..., S10:50
    order_cost = 80 + (store_idx % 3) * 20  # slight variation in fixed costs
    
    r, Q, cost = r_q_poisson_exact(
        holding_cost=store.local_holding_cost,      # 1.0
        stockout_cost=store.stockout_cost,          # 50.0  
        fixed_cost=order_cost,
        demand_mean=demand_mean,
        lead_time=1.0                               # from warehouse
    )
    
    store_policies[store_idx] = {'r': r, 'Q': Q, 'cost': cost, 'demand': demand_mean}
    print(f"{store.name}: r={r}, Q={Q}, weekly cost={cost:.1f}, demand={demand_mean}")

# Total expected cost across all stores
total_cost = sum(p['cost'] for p in store_policies.values())
print(f"\nTotal expected cost for 10 stores: ${total_cost:.1f}/week")


S1: r=39, Q=91, weekly cost=92.5, demand=38
S2: r=41, Q=102, weekly cost=103.1, demand=40
S3: r=44, Q=86, weekly cost=88.0, demand=42
S4: r=45, Q=98, weekly cost=99.5, demand=44
S5: r=47, Q=109, weekly cost=110.6, demand=46
S6: r=50, Q=92, weekly cost=94.1, demand=48
S7: r=51, Q=105, weekly cost=106.1, demand=50
S8: r=53, Q=116, weekly cost=117.5, demand=52
S9: r=56, Q=97, weekly cost=99.7, demand=54
S10: r=57, Q=111, weekly cost=112.3, demand=56

Total expected cost for 10 stores: $1023.4/week


In [10]:
# -------------------------- 
# 5. FULL MEIO: Working demo (no network object)
# --------------------------
from stockpyl.ssm_serial import optimize_base_stock_levels

print("\n" + "="*60)
print("STEP 5: MULTI-ECHELON OPTIMIZATION (serial demo)")
print("="*60)

# Direct parameter call (works with NumPy 2.0)
S_star, C_star = optimize_base_stock_levels(
    num_nodes=3,
    echelon_holding_cost=[2, 2, 3],
    lead_time=[2, 1, 1],
    stockout_cost=37.12,
    demand_mean=5,
    demand_standard_deviation=1
)

print("Optimal echelon base stocks:", {3-i: f"{v:.1f}" for i,v in enumerate(S_star.values())})
print("Total expected cost per week: ${C_star:.1f}\n")



STEP 5: MULTI-ECHELON OPTIMIZATION (serial demo)


ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.